In [22]:
import numpy as np
import pandas as pd

In [23]:
def extract_annotations(rawpath):
    """
    formats ANVIL-generated annotations into pandas dataframes
    """
    
    raw = pd.read_csv(rawpath, sep='\t')

    # keep only relevent columns
    df = raw[['Frame', 'Time', 'Scene info:narrative details - external', 'Scene info:narrative details - internal', 
             'Scene info:characters on screen', 'Scene info:Music Presence', 'speech:transcription', 
             'speech:character speaking', 'setting:indoor/outdoor', 'setting:setting', 
              'Scene name:scene name']]
    
    # drop duplicate annotations after first frame (drop consecutive duplicates only)
    df = df.loc[df['Scene info:narrative details - external'].shift() != df['Scene info:narrative details - external']]

    # rename columns
    df.columns = ['Frame', 'Onset time', 'Narrative details (external events)', 'Narrative details (internal state)', 
                 'Characters on screen', 'Music presence', 'Speech', 'Character speaking', 'Indoor/outdoor', 
                 'Setting', 'Scene name']

    # replace placeholder values with NaNs and preset booleans with corresponding strings
    df['Narrative details (internal state)'].replace(str(0),  np.nan, inplace=True)
    df['Music presence'].replace({0: 'no', 1: 'yes'}, inplace=True)
    df['Indoor/outdoor'].replace({1: 'indoor', 2: 'outdoor'}, inplace=True)
    for label in ['Characters on screen','Speech', 'Character speaking','Indoor/outdoor','Setting', 'Scene name']:
        df[label].replace([0, str(0), -1000, str(-1000)], np.nan, inplace=True)

    # find and drop rows for missed ms timepoints
    for index, row in df.iterrows():
        if row['Narrative details (external events)'] == str('-1000'):
            df.drop(index, inplace=True)

    df.reset_index(drop=True, inplace=True)
    
    return df

In [24]:
atlep1_df = extract_annotations('../../stimuli-annotations/atlanta-ep1-finished.txt')

In [25]:
atlep1_df.to_pickle('../../data/annotations_dfs/atlep1.p')

In [13]:
atlep2_df = extract_annotations('../../stimuli-annotations/atlanta-ep2-finished.txt')

/Users/paxtonfitzpatrick/anaconda/envs/py36/lib/python3.6/site-packages/IPython/core/interactiveshell.py:2802: DtypeWarning: Columns (4,5,8,9,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  if self.run_code(code, result):


In [14]:
atlep2_df.to_pickle('../../data/annotations_dfs/atlep2.p')

In [15]:
arrdev_df = extract_annotations('../../stimuli-annotations/arrdev-finished.txt')

In [16]:
arrdev_df.to_pickle('../../data/annotations_dfs/arrdev.p')